In [8]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [9]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("faiss.index")

In [10]:
with open("chunks.txt","r",encoding="utf-8") as f:
    chunks = f.readlines()

MODEL = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\benjo\anaconda3\envs\rag_faiss\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\benjo\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
def retrieve(query, k=3):
    q_emb = embed_model.encode(query).astype("float32")
    _, ids = index.search(q_emb.reshape(1,-1), k)
    return [chunks[i] for i in ids[0]]

def build_prompt(context, question):
    ctx = "\n".join(context)
    return f"""
Use ONLY the context to answer.
If unknown, say "I don't know".

Context:
{ctx}

Question:
{question}
"""

In [12]:
def answer_question(question):
    context = retrieve(question)
    prompt = build_prompt(context, question)
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=120)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, context

In [13]:
answer, ctx = answer_question("What is machine learning?")
print("ANSWER:\n", answer)

ANSWER:
 Response: Machine learning is a process of analyzing and analyzing data.
